# - 结构化绑定 auto [a,b] = func()
读 pair,map、结构体返回值，写 ROS 数据、解析结果巨爽

In [ ]:
#include 
#include 
#include 

std::tuple<int, double, std::string> get_user_data() {
return {101, 98.5, "Alice"};
}

int main() {
    // 现代 C++17 做法：一行代码直接拆解
    auto [id, score, name] = get_user_data();
    std::cout << name << " (ID: " << id << ") score: " << score << "\n";
    // 如果只想读，可以用 const auto& 减少拷贝
    const auto& [id_ref, score_ref, name_ref] = get_user_data();
}

# Fold Expressions
折叠表达式的一句话核心： 它允许你直接使用二元运算符（如 +, -, *, &&, , 等）将参数包在编译期原地“展开”并连环求值。  

| 语法形式       |	术语 |	编译器展开后的等价形式 (假设参数包为 1, 2, 3) |
| ------------- | ----- | --------------------------------------- |
| (... + pack)	| 一元左折叠	| ((1 + 2) + 3) |
| (pack + ...)	| 一元右折叠	| (1 + (2 + 3)) |
| (init + ... + pack)	| 二元左折叠	| (((init + 1) + 2) + 3) |
| (pack + ... + init)	| 二元右折叠	| (1 + (2 + (3 + init))) |  

`场景 A`：一元折叠（求所有参数的和）

In [ ]:
template<typename... Args>
auto sum(Args... args) {
    // 括号绝对不能省略！... 在运算符左边，代表左折叠
    return (... + args); 
}

int main() {
    return sum(1, 2, 3, 4); // 编译期直接变成：(((1 + 2) + 3) + 4)
}

`场景 B`：二元折叠（打印所有参数，并以逗号分隔，结合 , 运算符）

In [ ]:
#include <iostream>

template<typename... Args>
void print_all(Args... args) {
    // 利用逗号运算符：先打印空格，再打印元素。最后追加换行。
    ((std::cout << args << " "), ...);
    std::cout << "\n";
}

int main() {
    print_all(1, 3.14, "hello", 'c'); // 输出: 1 3.14 hello c 
}


## Python替代方案  

| **C++17 折叠表达式场景** | **C++17 核心写法** | **Python 最佳替代方案** |
|---|---|---|
| **通用任意二元运算** | `(... + pack)` | **`functools.reduce(lambda x, y: ..., pack)`** |
| **带初始值的运算** | `(init + ... + pack)` | **`functools.reduce(lambda x, y: ..., pack, init)`** |
| **全真才为真 (`&&`)** | `(pack && ...)` | **`all(pack)`** |
| **一真即为真 (`\|\|`)** | `(pack \|\| ...)` | **`any(pack)`** |
| **数值累加 (`+`)** | `(... + pack)` | **`sum(pack)`** |
| **参数包副作用连发 (`,` )** | `(func(pack), ...)` | **`for x in pack: func(x)`** 或列表推导式 |


In [1]:
import functools
print(functools.reduce(lambda x, y: x + y, [1, 2, 3, 4], 5))  # 输出: 15

15


# if constexpr  
它是编译期的 if 语句。编译器在编译时对条件求值（条件必须是编译期常量），只编译满足条件的那一个分支，不满足的分支会被编译器彻底抛弃，就像它从未存在过一样。

**彻底替代 std::enable_if 的经典示例**

`场景`：安全的数值递增（如果是指针，递增它指向的值；如果是普通数值，直接递增）

In [ ]:
#include <iostream>
#include <type_traits>

template<typename T>
void smart_increment(T& val) {
    // 条件必须在编译期就能确定
    if constexpr (std::is_pointer_v<T>) {
        if (val != nullptr) {
            (*val)++; // 如果 T 不是指针，这行代码在过去会引发编译报错
        }
    } else {
        val++;    // 如果 T 是指针，指针本身不能随便 val++，但因为分支被丢弃，所以绝对安全
    }
}

int main() {
    int a = 10;
    int* p = &a;

    smart_increment(a); // 实例化后，else 分支保留，a 变成 11
    smart_increment(p); // 实例化后，if 分支保留，*p (即 a) 变成 12
}


**当折叠表达式遇上 if constexpr：现代 C++ 元编程的巅峰组合**
这两个特性经常在处理类型复杂、数量可变的参数包时联手。例如，我们要写一个泛型的“类型序列检查函数”，判断传入的所有参数是否全部是整数 -> [code](./I_F/if_constexpr_fold_expr_main.cpp)

# Vocabulary Types (std::optional and std::variant)
| 特性 | `std::optional<T>` | `std::variant<T1, T2, ...>` |
| :--- | :--- | :--- |
| **核心概念** | **可能有一个值，也可能没有**（0 或 1 个） | **一定是多个指定类型中的某一个**（有且仅有 1 个） |
| **解决痛点** | 替代空指针、`std::pair<bool, T>` 或特殊错误码 | 替代传统 C 风格的 `union` 和 `void*` |
| **典型场景** | 数据库/字典查找可能失败、配置项可能未填写 | 解析 JSON（值可能是数字、字符串或布尔值） |
| **空状态** | 原生支持（用 `std::nullopt` 表示无值） | 默认初始化为第一个类型，或用 `std::monostate` 显式占位 |  
## std::optional

In [ ]:
#include <optional>
#include <iostream>

std::optional<int> find_user_age(int id) {
    if (id == 42) return 25; // 返回有效值
    return std::nullopt;     // 明确表示没有找到
}

int main() {
    auto age = find_user_age(42);
    
    // 检查是否有值
    if (age.has_value()) { 
        std::cout << "Age: " << *age << "\n"; // 解引用获取值
    }
    
    // 优雅的快捷方式：如果没有值，则返回默认值
    int final_age = age.value_or(18); 
}


## std::variant  
读取 `std::variant` 最推荐、最安全的方法是使用 `std::visit` 配合重载器（Overloaded lambdas）。

In [ ]:
#include <variant>
#include <iostream>
#include <string>

// 辅助工具：用于组合多个 lambda
template<class... Ts> struct overloaded : Ts... { using Ts::operator()...; };
template<class... Ts> overloaded(Ts...) -> overloaded<Ts...>;

int main() {
    // 变量可以是 int 或 std::string
    std::variant<int, std::string> data;

    data = 10;
    data = "Hello C++17";

    // 1. 通过 std::get 获取值（如果类型不匹配会抛出 std::bad_variant_access）
    // std::cout << std::get<int>(data); 

    // 2. 最佳实践：使用 std::visit 自动匹配类型
    std::visit(overloaded {
        [](int arg) { std::cout << "Int: " << arg << '\n'; },
        [](const std::string& arg) { std::cout << "String: " << arg << '\n'; }
    }, data);
}


# 内联变量 inline static
- 允许在类内直接定义并初始化静态成员变量，只需加上 `inline`。
- 编译器保证整个程序中该变量只有**一份定义**（即使头文件被多次包含）。

In [ ]:
class MyClass {
public:
    // C++17 起合法
    inline static int count = 0;           // 非 const 也可以
    inline static const double pi = 3.14;  // const 也可以
    inline static std::string name = "Grok"; // 复杂类型也可以
};

**关键要点**
- 必须加 inline  
只写 static int count = 0; 在类内仍然是声明 + 初始化尝试，在 C++17 下对非 const 仍不合法（除非是 constexpr）。
- inline static 与 static inline 顺序  
两种写法都可以，语义相同：  
```CPP
inline static int x = 1;
static inline int y = 2;   // 同样合法
```
- 与 constexpr 的关系  
constexpr 静态成员在 C++11/14 就可以在类内初始化。
C++17 的 inline 进一步放宽了非 constexpr 的情况。
也可以写成 inline static constexpr。

- 模板中的静态成员  
以前模板静态成员的定义非常麻烦，C++17 的 inline static 让头文件里直接写变得简单很多。
- 链接行为  
inline 变量遵循和 inline 函数类似的规则：可以在多个翻译单元中定义，但链接时合并成一份。

# 类模板参数推导 CTAD (Class Template Argument Deduction)
在 C++17 之前，实例化类模板时必须显式指定模板参数（例如 `std::vector<int> vec = {1, 2, 3};`）。而有了 CTAD 之后，编译器可以根据构造函数的实参自动推导出类模板的参数类型，就像函数模板参数推导一样方便。

In [ ]:
// before C++17
#include <utility>
#include <vector>

std::pair<int, double> p(1, 2.0); // 必须显式指定 <int, double>
std::vector<int> v = {1, 2, 3};   // 虽然有初值列表，但写法较为繁琐

// C++17 起，编译器可以自动推导类型
#include <utility>
#include <vector>

std::pair p(1, 2.0);        // 自动推导为 std::pair<int, double>
std::vector v = {1, 2, 3};  // 自动推导为 std::vector<int>

## 自定义类模板与 CTAD

CTAD 不仅适用于标准库容器（如 `std::vector`, `std::map`, `std::unique_ptr` 等），同样适用于我们自己编写的类模板。

In [ ]:
template <typename T>
class Box {
private:
    T value;
public:
    Box(T val) : value(val) {}
};

int main() {
    Box b(123); // 自动推导为 Box<int>
    // 在 C++17 之前，这会报错，必须写成 Box<int> b(123);
}

## 用户自定义推导指引（Deduction Guides）

有时，编译器的自动推导可能无法覆盖所有意图，或者推导出的类型不是你想要的。这时可以通过推导指引（Deduction Guides）来显式告知编译器如何推导。

In [ ]:
#include <iterator>
#include <vector>

template <typename T>
class Container {
public:
    // 1. 类内部的普通构造函数（保持不变）
    template <typename Iter>
    Container(Iter b, Iter e) {
        // ... 初始化逻辑
    }
};

// ==========================================
// 2. 如果直接 Container c(v.begin(), v.end()); 编译器可能无法正确推导 T 是什么
// 推导指引写在类定义外部（通常在紧挨着类下方的全局或命名空间作用域）
// ==========================================
template <typename Iter>
Container(Iter, Iter) -> Container<typename std::iterator_traits<Iter>::value_type>;


int main() {
    std::vector<int> v = {1, 2, 3, 4, 5};
    
    // 编译器看到这里时，会通过上面的推导指引，
    // 把迭代器指向的类型 (int) 提取出来，推导并实例化出 Container<int>
    Container c(v.begin(), v.end()); 
}

# std::filesystem
前身是 Boost.Filesystem
## 常用操作

In [ ]:
// 1. 路径拼接与处理
namespace fs = std::filesystem;

fs::path dir = "C:/my_folder";
fs::path file = dir / "sub" / "test.txt"; // 自动适配系统的路径分隔符

std::cout << file.filename() << "\n"; // 输出 "test.txt"
std::cout << file.extension() << "\n"; // 输出 ".txt"
std::cout << file.parent_path() << "\n"; // 输出 "C:/my_folder/sub"


In [ ]:
// 2. 文件状态与检查
fs::path p = "example.txt";

if (fs::exists(p)) {
    if (fs::is_regular_file(p)) {
        std::cout << "文件大小: " << fs::file_size(p) << " 字节\n";
    } else if (fs::is_directory(p)) {
        std::cout << "这是一个目录\n";
    }
}

In [ ]:
// 3. 目录与文件管理
fs::path dir = "test_dir";

// 创建目录
fs::create_directories(dir); // 类似 mkdir -p，支持递归创建父目录

// 复制文件
fs::copy_file("source.txt", "dest.txt", fs::copy_options::overwrite_existing);

// 重命名/移动文件
fs::rename("old_name.txt", "new_name.txt");

// 删除文件或空目录
fs::remove("dest.txt");

In [ ]:
// 4. 遍历目录
// fs::directory_iterator：单层遍历。
// fs::recursive_directory_iterator：递归遍历（深入所有子目录）。

// 递归遍历当前目录下的所有文件和文件夹
for (auto& entry : fs::recursive_directory_iterator(".")) {
    std::cout << entry.path() << '\n';
}

## 异常处理机制

`std::filesystem` 中的函数通常提供两种重载：

1. **抛出异常**：当出错（如文件不存在、权限不足）时抛出 fs::filesystem_error 异常。

2. **不抛出异常**：通过传入一个 std::error_code& ec 参数来捕获错误。

In [ ]:
std::error_code ec;
fs::remove("non_existent_file.txt", ec); // 如果出错，不会抛异常，错误码会记录在 ec 中
if (ec) {
    std::cout << "删除失败: " << ec.message() << '\n';
}